# Phase 1: DATA


In [9]:
# import section and parameters
from __future__ import annotations

import logging
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")
logger = logging.getLogger("ko_pep")

TICKERS: tuple[str, str] = ("KO", "PEP")
DATA_START = date(1990, 1, 1)
AS_OF = date(2026, 10, 9)          # last close used in the study (Friday); fixed for reproducibility
IS_END = date(2014, 12, 31)        # pre-registered split
OOS_START = date(2015, 1, 1)

RAW_PATH = Path("data/raw/prices_raw.pkl")
VALIDATED_PATH = Path("data/processed/prices_validated.pkl")


In [10]:
class DataFetchError(RuntimeError):
    """Raised when the data source returns no usable data."""


class StaleDataError(RuntimeError):
    """Raised when the last available close is older than expected."""

In [11]:
def fetch_prices(
    tickers: tuple[str, ...],
    start: date,
    as_of: date,
    path: Path,
    refresh: bool = False,
) -> pd.DataFrame:
    """Download dividend-adjusted daily prices, or load them from the raw checkpoint."""

    if path.exists() and not refresh:
        logger.info("Loading raw checkpoint %s", path)
        return pd.read_pickle(path)

    logger.info("Downloading %s from %s to %s", tickers, start, as_of)
    raw = yf.download(
        list(tickers),
        start=start.isoformat(),
        end=(as_of + timedelta(days=1)).isoformat(),
        auto_adjust=True,
        progress=False,
    )
    if raw is None or raw.empty:
        raise DataFetchError(f"No data returned for {tickers}")

    path.parent.mkdir(parents=True, exist_ok=True)
    raw.to_pickle(path)
    logger.info("Saved raw checkpoint %s (%d rows)", path, len(raw))
    return raw

In [ ]:
def validate_prices(
    raw: pd.DataFrame,
    tickers: tuple[str, ...],
    as_of: date,
    max_staleness_days: int = 5,
    jump_threshold: float = 0.20,
) -> pd.DataFrame:

    """
    Parameters
    ----------
    raw : pd.DataFrame
        Output of ``fetch_prices``.
    tickers : tuple[str, ...]
        Symbols to keep, in order.
    as_of : date
        No observation after this date is allowed.
    max_staleness_days : int, default 5
        Maximum calendar-day gap between ``as_of`` and the last close.
    jump_threshold : float, default 0.20
        Absolute daily log return above which a warning is logged.

    Returns
    -------
    pd.DataFrame
        Close prices indexed by date, one column per ticker, no missing values.

    Raises
    ------
    StaleDataError
        If the last close is older than ``max_staleness_days``."""


    close = raw["Close"].loc[:, list(tickers)].copy()
    close.index = pd.to_datetime(close.index)
    close = close.loc[close.index <= pd.Timestamp(as_of)]

    n_missing = close.isna().sum()
    if n_missing.any():
        logger.warning("Dropping rows with missing prices: %s", n_missing.to_dict())
    close = close.dropna(how="any")

    assert close.index.is_monotonic_increasing and close.index.is_unique, "Index must be sorted and unique"
    assert (close > 0).all().all(), "Prices must be strictly positive"
    assert close.index.max() <= pd.Timestamp(as_of), "No data after as_of"

    last_close = close.index.max().date()
    if (as_of - last_close).days > max_staleness_days:
        raise StaleDataError(f"Last close {last_close} is older than {max_staleness_days} days before {as_of}")

    log_ret = np.log(close).diff().dropna()
    jumps = log_ret[(log_ret.abs() > jump_threshold).any(axis=1)]
    if not jumps.empty:
        logger.warning("Large daily moves to inspect:\n%s", jumps)

    return close

In [5]:
raw = fetch_prices(TICKERS, DATA_START, AS_OF, RAW_PATH)
prices = validate_prices(raw, TICKERS, AS_OF)

raw_flat = raw.copy()
raw_flat.columns = [f"{ticker}_{field}" for field, ticker in raw_flat.columns]
raw_flat.to_csv(RAW_PATH.with_suffix(".csv"), index_label="date")

VALIDATED_PATH.parent.mkdir(parents=True, exist_ok=True)
prices.to_pickle(VALIDATED_PATH)
prices.to_csv(VALIDATED_PATH.with_suffix(".csv"), index_label="date", float_format="%.6f")

is_mask = prices.index <= pd.Timestamp(IS_END)
prices_is = prices.loc[: pd.Timestamp(IS_END)]
assert prices_is.index.max() <= pd.Timestamp(IS_END), "In-sample frame must not contain OOS dates"
oos_mask = prices.index >= pd.Timestamp(OOS_START)
assert is_mask.sum() + oos_mask.sum() == len(prices), "IS and OOS must cover all rows exactly once"

summary = pd.DataFrame(
    {
        "first date": [prices.index.min().date()],
        "last date": [prices.index.max().date()],
        "observations": [len(prices)],
        "in-sample obs": [int(is_mask.sum())],
        "out-of-sample obs": [int(oos_mask.sum())],
    }
)
summary

2026-10-10 10:47:12,319 ko_pep INFO Loading raw checkpoint data\raw\prices_raw.pkl


,first date,last date,observations,in-sample obs,out-of-sample obs
0,1990-01-02,2026-10-09,9261,6301,2960


In [6]:
normalised = prices_is / prices_is.iloc[0] * 100

fig = go.Figure()
for ticker in TICKERS:
    fig.add_trace(go.Scatter(x=normalised.index, y=normalised[ticker], name=ticker, mode="lines"))
fig.update_layout(
    title="KO and PEP, dividend-adjusted, rebased to 100 (log scale) in-sample 1990–2014",
    yaxis_type="log",
    yaxis_title="Index (start = 100)",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

In [7]:
ratio = prices_is["KO"] / prices_is["PEP"]

fig = go.Figure(go.Scatter(x=ratio.index, y=ratio, mode="lines", name="KO / PEP"))
fig.update_layout(
    title="Price ratio KO / PEP in-sample 1990–2014",
    yaxis_title="Ratio",
    template="plotly_white",
)
fig.show()

Three regimes are observed:
- 1990–1998 the ratio grows from 0.5 to 1 approximately, with a peak in August 1998
- 1998-2005 the ratio decreases to 0.35/0.40 cancelling the previous growth
- 2005-2014 the ratio steadily oscillates between 0.35 and 0.55

In [8]:
log_ret = np.log(prices_is).diff().dropna()

periods = {"In-sample (1990–2014)": log_ret}
for decade, chunk in log_ret.groupby((log_ret.index.year // 10) * 10):
    periods[f"{decade}s"] = chunk

corr_table = pd.DataFrame(
    {
        "correlation": [chunk["KO"].corr(chunk["PEP"]) for chunk in periods.values()],
        "observations": [len(chunk) for chunk in periods.values()],
    },
    index=list(periods.keys()),
).round(3)
assert corr_table["correlation"].between(-1, 1).all()
assert log_ret.index.max() <= pd.Timestamp(IS_END), "Correlation must use in-sample data only"

rolling_corr = log_ret["KO"].rolling(252).corr(log_ret["PEP"])

fig = go.Figure(go.Scatter(x=rolling_corr.index, y=rolling_corr, mode="lines", name="1-year rolling correlation"))
fig.update_layout(
    title="1-year rolling correlation of daily log returns, KO vs PEP in-sample 1990–2014",
    yaxis_title="Correlation",
    yaxis_range=[-0.2, 1],
    template="plotly_white",
)
fig.show()
corr_table

,correlation,observations
In-sample (1990–2014),0.514,6300
1990s,0.508,2527
2000s,0.512,2515
2010s,0.602,1258


The return correlation is always positive but clearly time-varying. The full in-sample value is 0.51, but the 1-year rolling estimate ranges from about
0.2 to 0.8. 

The 2008–09 peak (~0.75) is probably a market-wide effect, since correlations rise across
most stocks in a crisis. 

From 2008 to 2014 the correlation stays higher (0.55–0.7), in line with the more stable ratio.
